In [1]:
import pandas as pd
import torch

In [2]:
print("GPU available: ", torch.cuda.is_available())

GPU available:  False


In [3]:
df = pd.read_csv("../data/clean_post.csv")

In [4]:
from transformers import pipeline
#!pip install transformers

In [5]:
sentiment_model = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    max_length = 512,
    device = 0);

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [6]:
def get_sentiment(text):
    try:
        result = sentiment_model(str(text)[:512])[0]
        return result["label"], round(result["score"], 4)
    except:
        return "neutral", 0.0

In [7]:
print("Running sentiment analysis... this will take a few minutes")
df[["sentiment", "confidence"]] = df["clean_text"].apply(
    lambda x: pd.Series(get_sentiment(x))
)

Running sentiment analysis... this will take a few minutes


In [8]:
print("\nSentiment Distribution: ")
print(df["sentiment"].value_counts())
print("\nAverage confidence per sentiment: ")
print(df.groupby("sentiment")["confidence"].mean().round(3))
df.to_csv("../data/sentiment_posts.csv")


Sentiment Distribution: 
sentiment
negative    1311
neutral      661
positive     119
Name: count, dtype: int64

Average confidence per sentiment: 
sentiment
negative    0.748
neutral     0.682
positive    0.703
Name: confidence, dtype: float64


In [9]:
print(df.head())

   Unnamed: 0       id                                              title  \
0           0  1rlblhn  President Bought Netflix Debt in January 2026,...   
1           1  1rl6lay  US air defenses may not be able to intercept m...   
2           2  1rlhffx  GOP state lawmakers urge White House to halt e...   
3           3  1rkv7az  Brendan Carr Can’t Explain Why ‘Equal Time’ Ru...   
4           4  1rlejk6  I was at a QuitGPT protest, and the discontent...   

  body  score  upvote_ratio  num_comments                    flair  \
0  NaN   8871          0.97           568                 Business   
1  NaN   5598          0.95           814                 Business   
2  NaN    732          0.97            52  Artificial Intelligence   
3  NaN  23944          0.96           401                 Politics   
4  NaN    790          0.96            22  Artificial Intelligence   

                 author                                                url  \
0            ControlCAD  https://www.h